### create dataset with some nulls

In [0]:
data = [
    (1, "A", None),
    (2, "B", 5000),
    (3, None, 7000),
    (4, "D", None)
]
df = spark.createDataFrame(data, ["id", "name", "salary"])

In [0]:
df.display()
df.printSchema()

### isNull() / isNotNull()

In [0]:
df.filter(df.salary.isNull()).display()

### fillna()

In [0]:
df.fillna({"salary": 0}).display()

### dropna()

In [0]:
df.dropna().display()

### coalesce()

In [0]:
from pyspark.sql.functions import coalesce, lit

df.withColumn("salary_new", coalesce(df.salary, lit(0))).display()

### Student Task
### Replace NULL salary with avg salary
### Drop rows where name is NULL
### Count NULL records

In [0]:
from pyspark.sql.functions import avg


avg_salary = df.select(avg("salary")).first()[0]


df_neww = df.fillna({"salary": avg_salary})

df_neww.display()

In [0]:

df.dropna(subset=["name"]).display()

In [0]:
count_null_records=df.filter(df.salary.isNull()).count()
print(count_null_records)

### Type Casting + Column Operations

### Casting

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("salary", df.salary.cast("int"))
df.printSchema()

### Column Operations

In [0]:
from pyspark.sql.functions import col, when

df.select("id", "salary").display()

df.withColumn("bonus", col("salary") * 0.1).display()

df.withColumn("category",
    when(col("salary") > 6000, "High")
    .otherwise("Low")
).display()

### Replace the above with below

In [0]:
# Step 1: Replace NULL salary with 1000
df = df.withColumn(
    "salary",
    when(col("salary").isNull(), 1000)
    .otherwise(col("salary"))
)

# Step 2: Add bonus column (10% of salary)
df = df.withColumn(
    "bonus",
    col("salary") * 0.1
)

# Step 3: Add category column
df = df.withColumn(
    "category",
    when(col("salary") > 6000, "High")
    .otherwise("Low")
)

# Final output
print("Final Data:")
df.display()

### Student Task
### Create new column: tax = salary * 0.2
### Create flag: is_high_salary
### Rename column

In [0]:
def grade(salary):
    if salary is None:
        return "Unknown"
    elif salary > 6000:
        return "A"
    else:
        return "B"

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

grade_udf = udf(grade, StringType())

In [0]:
df = df.withColumn("grade", grade_udf(df.salary))
df.display()

### Same logic WITHOUT UDF:

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "grade",
    when(col("salary").isNull(), "Unknown")
    .when(col("salary") > 6000, "A")
    .otherwise("B")
)

### This is: Faster, Optimized, Recommended

### WHEN SHOULD YOU USE UDF?
### Only when: Complex logic, No built-in alternative, Custom transformations